# Visualize 2E Data. The used data is different from the exported data of the models

### models

In [1]:
#import libraries

import numpy as np

import itertools
import os
import datetime
import matplotlib.pyplot as plt
import matplotlib
%matplotlib qt

import pickle
# import mgzip
import pandas as pd
import importlib

import scipy

import subprocess
from subprocess import call
# from xlsx2csv import Xlsx2csv


In [9]:
rootdir = r"C:\Users\Arbin\source\repos\maggyo\Arbin-CTI\Output_Data\visualization"
cell_num = 1

#LOAD IN 2E DATA


#structure to store data
masterInput = []

for subdir, dirs, files in os.walk(rootdir):

#   real_data = []

#     strpath = subdir
#     idx = strpath.rfind("channel_")
#     if(idx == -1):
#         continue
    
#     text = strpath[idx+7:idx+10]
#     sheet_num  = ""
#     for i in text:
#         if i.isdigit():
#             sheet_num += i

#     # print(subdir)
#     temp2 = strpath.rfind(rootdir)
#     temp3 = strpath.find('C\Jiayi')
#     c_rate = strpath[temp2+len(rootdir)+1:temp3]
#     if(len(c_rate) > 5):
#         c_rate = "stepped"
    
    appended = False
#     temp = 1
#     print(cell_num, c_rate)
    for file in files:
        if "archive" in os.path.join(subdir, file):
            continue

    
        strpath = os.path.join(subdir, file)
        
        #If past cells are need to be plotted, I can do either # THE PART BELOW or copy the past cell to the directory of running cells
        #temp = os.path.getmtime(strpath)
        #dt_m = datetime.datetime.fromtimestamp(temp)
        #compare_date =  datetime.date.today()


        #if(dt_m < compare_date):
            #continue


        print(strpath)

        temp2 = strpath.rfind('_')+1
        temp3 = strpath.rfind('.')
        channel_num = strpath[temp2:temp3]

        real_data = pd.read_csv(strpath, delimiter="\t\t\t", names=['Time', 'Step_Index', 'cycle_num', 'Voltage', 'Current', 'NEP'])
        real_data['cell_num'] = cell_num
        real_data['channel_num'] = channel_num
 
        masterInput.append(real_data)
        cell_num+=1

print("complete")
masterInput = pd.concat(masterInput)
masterInput.set_index(['cell_num', 'cycle_num'], inplace=True)

C:\Users\Arbin\source\repos\maggyo\Arbin-CTI\Output_Data\visualization\threeEandTwoE_charging_output_09262023_Channel_12.txt


C:\Users\Arbin\AppData\Local\Temp\ipykernel_13368\1344299877.py:58: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  real_data = pd.read_csv(strpath, delimiter="\t\t\t", names=['Time', 'Step_Index', 'cycle_num', 'Voltage', 'Current', 'NEP'])


complete


In [10]:
masterInput.index.get_level_values(0)

Int64Index([1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
            ...
            1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
           dtype='int64', name='cell_num', length=474290)

In [11]:
temp = masterInput.groupby(['cell_num', 'cycle_num']).apply(lambda x: x.Time - x.Time.min())
temp = temp.droplevel(0)
temp = temp.droplevel(0)
temp

masterInput['Time'] = temp

In [15]:
masterInput = masterInput[masterInput['NEP']<0.5]
masterInput = masterInput[masterInput['Voltage']>3.32]
cells = masterInput.index.get_level_values(0).unique()

masterInput.NEP.min()

In [16]:
plt.close('all')

In [21]:
#plot cycle to cycle data for every cell
use_cols = ['Time']
from functions import DataPlotting
importlib.reload(DataPlotting)

for i in cells:
    # i = 8
    try:
        DataPlotting.plot_multiple_cycles_one_cell(masterInput.drop(['NEP']), masterInput['NEP'], i, use_cols, 'channel ' + masterInput.loc[i].channel_num.unique())
    except Exception as e:
        print('failed ' + str(i))
        print(e)
    # break

c:\Users\Arbin\source\repos\maggyo\Arbin-CTI\functions\DataPlotting.py:71: UserWarning: cannot plot cell #1, cycle 1
  warnings.warn("cannot plot cell #" + str(cellNum) + ", cycle " + str(cycle))
c:\Users\Arbin\source\repos\maggyo\Arbin-CTI\functions\DataPlotting.py:71: UserWarning: cannot plot cell #1, cycle 2
  warnings.warn("cannot plot cell #" + str(cellNum) + ", cycle " + str(cycle))
c:\Users\Arbin\source\repos\maggyo\Arbin-CTI\functions\DataPlotting.py:71: UserWarning: cannot plot cell #1, cycle 3
  warnings.warn("cannot plot cell #" + str(cellNum) + ", cycle " + str(cycle))


In [24]:
cols = ['Time','Voltage','Current','Step_Index']

from functions import DataPlotting
importlib.reload(DataPlotting)

DataPlotting.plot_one_cycle_multiple_cells(masterInput.drop(['NEP']), masterInput['NEP'], 4, cells, [], cols)